# Free text-to-video with Google Colab

This notebook uses the openly licensed `zai-org/CogVideoX-2b` model. It does not use fal.ai, an API key, or paid API credits.

1. Open this notebook in Google Colab.
2. Choose **Runtime > Change runtime type > T4 GPU** if it is offered. Do not choose a paid runtime if you want to stay on the free tier.
3. Run the cells from top to bottom, edit the prompt, then download `output.mp4`.

Free GPU availability and session time are limited and not guaranteed. Generation can be slow or run out of memory. This produces a 720x480, 49-frame clip (about 6 seconds at 8 fps). It is a manual notebook, not an always-on API for the studio app.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is attached. In Colab, choose Runtime > Change runtime type > T4 GPU, "
        "if available, then reconnect and rerun this cell."
    )

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())

In [ ]:
%pip install -q -U "diffusers>=0.30.1" transformers accelerate imageio-ffmpeg

In [ ]:
import torch
from diffusers import CogVideoXPipeline

MODEL_ID = "zai-org/CogVideoX-2b"
pipe = CogVideoXPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
)
pipe.enable_model_cpu_offload()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()
print("Loaded", MODEL_ID)

If the free GPU runs out of memory, restart the runtime and replace `enable_model_cpu_offload()` above with `enable_sequential_cpu_offload()`. Sequential offload uses less GPU memory but can be much slower.

In [ ]:
prompt = "A small red kite flying over a green field on a breezy afternoon, cinematic camera movement, soft natural light."

result = pipe(
    prompt=prompt,
    height=480,
    width=720,
    num_frames=49,
    num_inference_steps=50,
    guidance_scale=6,
    generator=torch.Generator(device="cuda").manual_seed(42),
)
frames = result.frames[0]
print("Generated", len(frames), "frames")

In [ ]:
from diffusers.utils import export_to_video
from IPython.display import Video, display

output_path = "/content/output.mp4"
export_to_video(frames, output_path, fps=8)
display(Video(output_path, embed=True))

In [ ]:
from google.colab import files

files.download(output_path)